# Case study: a drift report that turned out to be a level shift

Support says churn "went up in June." The churn table is a weekly
snapshot: tenure, a churn score, and the snapshot date. The question is
not whether the numbers moved — they did — but *what kind* of change it
was: a slow trend the model should learn, or a break the pipeline should
investigate. Those have different fixes, and the profiler has a finding
category whose whole job is telling them apart: `changes/residual_level`.

The fixture plants the classic upstream story: a score that depends on
tenure, gains a flat +0.15 from a specific date, and a snapshot calendar
that lost one week along the way.

In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(88)

base = dt.date(2024, 1, 8)
missing_week = dt.date(2024, 5, 20)                  # one snapshot week skipped
snapshot_date = []
for w in range(30):
    d = base + dt.timedelta(weeks=w)
    if d == missing_week:
        continue
    snapshot_date.extend([d] * 200)       # 200 rows per week
n = len(snapshot_date)

tenure = [rng.randrange(1, 60) for _ in range(n)]
score = [round(0.3 + 0.01 * t + rng.uniform(-0.05, 0.05), 4) for t in tenure]
# Level shift from 2024-06-03: the score gains +0.15 everywhere after that date.
cutoff = snapshot_date.index(dt.date(2024, 6, 3))
churn_score = [
    round(score[i] + (0.15 if i >= cutoff else 0.0) + rng.uniform(-0.02, 0.02), 4)
    for i in range(n)
]

df = pl.DataFrame({
    "snapshot_date": snapshot_date,
    "tenure": tenure,
    "churn_score": churn_score,
})

report = onceover.profile(df)
print(report.overview["headline"])

Before reading anything interesting, the calendar gets audited — temporal
claims are only meaningful against the cadence they assume:

In [ ]:
for f in report.findings:
    if f["category"] in {"gaps", "intervals", "ordering"}:
        print(f"[{f['category']:9s}] {f['claim']}")

The snapshot column runs weekly (modal cadence 7 days), non-decreasing
with row order — a time-ordered panel, which is exactly the structure the
residual-shift probe needs — and the calendar itself has a hole: one
missing week, counted against the weekly cadence. Now the interesting
part:

In [ ]:
changes = [f for f in report.findings
            if f["category"] == "changes" and f["subject"].get("target") == "churn_score"]
print("residual-shift findings for churn_score:", len(changes))
for f in changes:
    m = f["evidence"][0]["metrics"]
    print(f["claim"])
    print()
    print("   breakpoint:", m["breakpoint"])
    print("   residual mean before/after:",
          round(m["residual_mean_before"], 4), "/", round(m["residual_mean_after"], 4))
    print("   holdout error reduction:", f"{100 * m['holdout_error_reduction']:.1f}%")
    print("   reference model: churn_score ~ tenure +",
          m["reference_model"]["coefficients"])

## Reading the finding

The reference model is the honest part: `churn_score` on `tenure`, a
plain linear fit whose coefficients are recorded in the finding. Its
residuals, ordered by the snapshot date, step once — the centered
cumulative-sum scan puts the breakpoint at 2024-05-27, inside the missing
week's empty slot, so the offset applies from the 2024-06-03 block
onward — and a fixed before/after offset correction cuts holdout squared
error by 83.9%. So the June increase is not a trend the model missed; it
is a *level* in the score that the tenure relationship does not explain.

For the pipeline this means:

- **Do not let the model silently absorb it.** A retrained model will
  flatten the shift into its coefficients and misprice every tenure band
  before June.
- **Add the period as a feature or split the panel.** Either a
  `post_june` indicator or two models; the finding's breakpoint gives the
  split date, and the offsets give the expected magnitude (about 0.15
  score points).
- **Ask what happened on that date.** A level shift in a derived score is
  usually upstream: a scoring formula change, a source system swap, a
  definition edit. The profiler measures *that the residuals changed*;
  the cause is organizational archaeology.

The verification is cheap and worth doing explicitly — monthly means,
straight from the table:

In [ ]:
with pl.Config(tbl_rows=40, tbl_width_chars=90):
    monthly = df.group_by(
        pl.col("snapshot_date").dt.strftime("%Y-%m").alias("month")
    ).agg(pl.col("churn_score").mean().round(4).alias("mean_score")).sort("month")
    print(monthly)

The monthly means tell the same story as the finding — a step, not a
slope — with the finding contributing the exact date, the offsets, and the
reference model that defines "residual."

## What the finding does not claim

No p-value is attached: this is a measured level change under a stated
model, on disclosed train/holdout rows, from a pass that tests *one*
breakpoint on the *first* date column. Multiple breaks, gradual ramps,
and breaks driven by a column other than the first temporal one are out
of scope today ([](../appendix/directions.ipynb)). A skeptical read also
keeps the calendar finding in the same email as the drift question — the
missing week of May 20 belongs to whoever owns the snapshot job.

The general pattern: when a stakeholder reports "it changed," the first
profile answers *what kind of change* — and the second conversation
starts from a measured breakpoint instead of an impression.